# RAG Performance & Fairness Evaluation Toolkit (OpenVINO + LangChain)

This notebook demonstrates how to build and evaluate a Retrieval-Augmented Generation (RAG) pipeline using OpenVINO™ for accelerated performance on Intel hardware. We will use Hugging Face and LangChain libraries to construct the pipeline.

## Process Overview

The process involves:

1.  **[Environment Setup](#1-Environment-Setup)**: Installing necessary libraries.
2.  **[LLM and Tokenizer Setup](#2-LLM-and-Tokenizer-Setup)**: Loading a language model (Microsoft's Phi-3-mini) and its tokenizer, optimized with OpenVINO.
3.  **[Embedding Model Setup](#3-Embedding-Model-Setup)**: Preparing an embedding model to convert text into vector representations.
4.  **[Data Loading and Processing](#4-Data-Loading-and-Processing)**: Fetching documents from a web source, splitting them into manageable chunks, and creating vector embeddings.
5.  **[Vector Store and Retriever Setup](#5-Vector-Store-and-Retriever-Setup)**: Storing the embeddings in a ChromaDB vector store and setting up a retriever with reranking for improved accuracy.
6.  **[Building the RAG Chain](#6-Building-the-RAG-Chain)**: Creating a `RetrievalQA` chain that combines the retriever and the LLM.
7.  **[Running the RAG Pipeline](#7-Running-the-RAG-Pipeline)**: Asking a question to get a response from the RAG system.
8.  **[Evaluation](#8-Evaluation)**: Using a comprehensive `OpenVINORAGEvaluator` to assess the quality of the generated response based on various metrics like BLEU, ROUGE, BERTScore, perplexity, and bias.

#### Table of contents:

- [1. Environment Setup](#1.-Environment-Setup)
- [2. LLM and Tokenizer Setup](#2.-LLM-and-Tokenizer-Setup)
    - [Create a LangChain-compatible LLM Pipeline](#Create-a-LangChain-compatible-LLM-Pipeline)
- [3. Embedding Model Setup](#3.-Embedding-Model-Setup)
- [4. Data Loading and Processing](#4.-Data-Loading-and-Processing)
    - [Split Documents into Chunks](#Split-Documents-into-Chunks)
- [5. Vector Store and Retriever Setup](#5.-Vector-Store-and-Retriever-Setup)
    - [Add Documents to the Vector Store](#Add-Documents-to-the-Vector-Store)
    - [Set up a Reranking Retriever](#Set-up-a-Reranking-Retriever)
- [6. Building the RAG Chain](#6.-Building-the-RAG-Chain)
- [7. Running the RAG Pipeline](#7.-Running-the-RAG-Pipeline)
    - [Extract Answer and Context for Evaluation](#Extract-Answer-and-Context-for-Evaluation)
- [8. Evaluation](#8.-Evaluation)
    - [Run the Evaluation](#Run-the-Evaluation)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/llm-rag-langchain/llm-rag-langchain-eval.ipynb" />

## 1. Environment Setup
[back to top ⬆️](#Table-of-contents:)


First, let's ensure all the required Python packages are installed. The following commands handle the installation of essential libraries. These are typically only needed if you encounter version conflicts or issues with existing installations.

In [ ]:
import os
import requests
from pathlib import Path
import io

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    with open("notebook_utils.py", "w") as f:
        f.write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

if not Path("text_example_en.pdf").exists():
    text_example_en = "https://github.com/openvinotoolkit/openvino_notebooks/files/15039728/Platform.Brief_Intel.vPro.with.Intel.Core.Ultra_Final.pdf"
    r = requests.get(url=text_example_en)
    content = io.BytesIO(r.content)
    with open("text_example_en.pdf", "wb") as f:
        f.write(content.read())

from pip_helper import pip_install

os.environ["GIT_CLONE_PROTECTION_ACTIVE"] = "false"

pip_install("-q", "--pre", "-U", "openvino==2025.4.0", "--extra-index-url", "https://storage.openvinotoolkit.org/simple/wheels/nightly")
pip_install("-q", "--pre", "-U", "openvino-tokenizers==2025.4.0.0", "--extra-index-url", "https://storage.openvinotoolkit.org/simple/wheels/nightly")
pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "--upgrade-strategy",
    "eager",
    "optimum[openvino,nncf,onnxruntime]",
    "optimum-intel==1.26.1",
    "sacrebleu==2.5.1",
    "rouge-score==0.1.2",
    "nncf==2.19.0",
    "bert-score==0.3.13",
    "transformers==4.53.3",
    "onnx",
    "nltk==3.9.2",
    "numpy==2.2.6",
    "langchain==0.3.27",
    "langchain_community==0.3.31",
    "chromadb==1.3.5",
    "langchain-chroma==0.2.6",
    "langchain-huggingface==0.3.1",
    "sentence-transformers==5.1.2",
    "Flashrank==0.2.10",
    "python-docx==1.2.0",
    "huggingface-hub==0.36.0",
    "faiss-cpu==1.13.1",
    "pypdf==6.4.1",
    "torch==2.8.0",
    "tokenizers==0.21.4",
)
# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("llm-rag-langchain-eval.ipynb")

## 2. LLM and Tokenizer Setup
[back to top ⬆️](#Table-of-contents:)


Next, we load the Large Language Model (LLM) and its corresponding tokenizer. We use `optimum-intel` to load a pre-converted OpenVINO model. In this example, we use `OpenVINO/Phi-3-mini-4k-instruct-int4-ov`, but you can replace it with another compatible model.

- **`OVModelForCausalLM`**: Loads a causal language model in the OpenVINO format from a pre-converted checkpoint.

In [2]:
from notebook_utils import device_widget

llm_device = device_widget()

llm_device

Dropdown(description='Device:', index=3, options=('CPU', 'GPU', 'NPU', 'AUTO'), value='AUTO')

In [ ]:
import warnings

warnings.filterwarnings("ignore")
from optimum.intel import OVModelForCausalLM
from transformers import AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline

# Load pre-converted OpenVINO model
model = OVModelForCausalLM.from_pretrained(
    "OpenVINO/Phi-3-mini-4k-instruct-int4-ov",
    device=llm_device.value,
)

tokenizer = AutoTokenizer.from_pretrained("OpenVINO/Phi-3-mini-4k-instruct-int4-ov")

C:\Users\Local_Admin\rag_issue\lib\site-packages\openvino\runtime\__init__.py:10: DeprecationWarning: The `openvino.runtime` module is deprecated and will be removed in the 2026.0 release. Please replace `openvino.runtime` with `openvino`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


INFO:nncf:Statistics of the bitwidth distribution:
+---------------------------+-----------------------------+----------------------------------------+
| Weight compression mode   | % all parameters (layers)   | % ratio-defining parameters (layers)   |
+===========================+=============================+========================================+
| int8_asym, per-channel    | 100% (130 / 130)            | 100% (130 / 130)                       |
+---------------------------+-----------------------------+----------------------------------------+


Output()

### Create a LangChain-compatible LLM Pipeline
[back to top ⬆️](#Table-of-contents:)


We now create a `text-generation` pipeline using the OpenVINO-optimized model and tokenizer. This pipeline is then wrapped in `HuggingFacePipeline` to make it compatible with the LangChain ecosystem. A quick test is run to confirm the pipeline is working correctly.

In [4]:
# Create a text-generation pipeline with the OpenVINO model
llm_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, device=model.device, max_new_tokens=100, top_k=50, temperature=0.1, do_sample=True)

# Create a LangChain instance from the Hugging Face pipeline
llm = HuggingFacePipeline(pipeline=llm_pipeline)

Device set to use cpu


What is an ocean? An ocean is a vast body of saltwater that covers approximately 71% of the Earth's surface. It is the largest component of the hydrosphere and plays a crucial role in regulating the planet's climate and supporting a diverse range of marine life. Oceans are divided into five major basins: the Pacific, Atlantic, Indian, Southern (Antarctic), and Arctic Oceans. These bodies of water are interconnected and are characterized by their


## 3. Embedding Model Setup
[back to top ⬆️](#Table-of-contents:)


For the retrieval part of our RAG pipeline, we need an embedding model to convert text documents into numerical vectors. We use `OpenVINOBgeEmbeddings` from `langchain_community`, which provides OpenVINO-optimized embeddings for efficient performance. Here, we use the `bge-small-en-v1.5` model.

In [5]:
from notebook_utils import device_widget

embedding_device = device_widget()

embedding_device

Dropdown(description='Device:', index=3, options=('CPU', 'GPU', 'NPU', 'AUTO'), value='AUTO')

In [ ]:
from langchain_community.embeddings import OpenVINOBgeEmbeddings
from sentence_transformers import SentenceTransformer
import os
import warnings

warnings.filterwarnings("ignore")
# First time: Download and save the model
embedding_model_name = "BAAI/bge-small-en-v1.5"  # Full HF repo path
save_directory = "./saved_bge_model"

# Download the model using SentenceTransformer directly
st_model = SentenceTransformer(embedding_model_name)
st_model.save(save_directory)
print(f"Model saved to {save_directory}")

# Now create the OpenVINO embedding with the saved model
embedding = OpenVINOBgeEmbeddings(
    model_name_or_path=save_directory,  # Use saved path
    model_kwargs={"device": embedding_device.value},
    encode_kwargs={"normalize_embeddings": True},
)

# Load the saved model from local directory
local_model_path = "./saved_bge_model"

embedding = OpenVINOBgeEmbeddings(
    model_name_or_path=local_model_path,
    model_kwargs={"device": embedding_device.value},
    encode_kwargs={"normalize_embeddings": True},
)

# Test the loaded model
text = "This is a test document."
embedding_result = embedding.embed_query(text)
print("Sample embedding (first 3 dimensions):", embedding_result[:3])

Model saved to ./saved_bge_model
Sample embedding (first 3 dimensions): [-0.042086612433195114, 0.06681863963603973, 0.007916754111647606]


## 4. Data Loading and Processing
[back to top ⬆️](#Table-of-contents:)


Now we'll load the documents that will form the knowledge base for our RAG pipeline. This notebook includes two methods for loading documents:

1.  **Web Crawling (Enabled by default)**: Fetches content from a website's sitemap. We use `WebBaseLoader` to load content from URLs found in the sitemap of Zerodha Varsity.
2.  **Local File Loading (Commented out)**: A robust `LangChainDocumentLoader` class is provided to load various file types (`.txt`, `.pdf`, `.docx`, etc.) from a local directory. You can uncomment and adapt this section if you want to use your own local files.

In [ ]:
import warnings

warnings.filterwarnings("ignore")
'''
# --- Method 1: Load documents by crawling a web page (default) ---
from langchain_community.document_loaders import WebBaseLoader
import ssl
from urllib.request import Request, urlopen
import bs4
from bs4 import BeautifulSoup
def get_sitemap(url):
    """Fetches and parses an XML sitemap from a URL."""
    req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    response = urlopen(req)
    xml = BeautifulSoup(response, "lxml-xml", from_encoding=response.info().get_param("charset"))
    return xml

def get_urls_from_sitemap(xml):
    """Extracts all URLs from a parsed sitemap XML."""
    urls = [loc.text for loc in xml.find_all("loc")]
    return urls

# Bypass SSL verification issues if they arise
ssl._create_default_https_context = ssl._create_stdlib_context

sitemap_url = "https://zerodha.com/varsity/chapter-sitemap2.xml"
sitemap_xml = get_sitemap(sitemap_url)
urls = get_urls_from_sitemap(sitemap_xml)

# Load documents from the collected URLs
docs = []
for i, url in enumerate(urls):
    try:
        loader = WebBaseLoader(url)
        docs.extend(loader.load())
        if (i + 1) % 10 == 0:
            print(f"Loaded {i + 1}/{len(urls)} URLs")
    except Exception as e:
        print(f"Failed to load {url}: {e}")

print(f"\nTotal documents loaded: {len(docs)}")
'''
# --- Method 2: Load a single document from the system ---

import os
from langchain.document_loaders import (
    TextLoader,
    PyPDFLoader,
)
from langchain.schema import Document as LCDocument
from typing import List


class SingleDocumentLoader:
    """Load a single document from the file system."""

    def __init__(self, file_path: str):
        self.file_path = file_path

    def load(self) -> List[LCDocument]:
        """Loads a single document based on file type."""
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"File not found: {self.file_path}")

        # Determine file type and load accordingly
        if self.file_path.endswith(".pdf"):
            loader = PyPDFLoader(self.file_path)
        elif self.file_path.endswith(".txt"):
            loader = TextLoader(self.file_path, encoding="utf-8")
        else:
            raise ValueError(f"Unsupported file type: {self.file_path}")

        return loader.load()


# Usage Example:
loader = SingleDocumentLoader(file_path="text_example_en.pdf")  # Change to your file path
docs = loader.load()
print("Loaded document(s) from single file.")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded document(s) from single file.


### Split Documents into Chunks
[back to top ⬆️](#Table-of-contents:)


LLMs have a limited context window, so we need to split large documents into smaller chunks. This ensures that the model can process the retrieved information effectively. We use `RecursiveCharacterTextSplitter` which is a smart way to split text while trying to keep related content together.

In [8]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Split the documents into smaller chunks with a specified size and overlap
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1250, chunk_overlap=100, length_function=len, is_separator_regex=False)

split_docs = text_splitter.split_documents(docs)
print(f"Documents split into {len(split_docs)} chunks.")

Documents split into 6 chunks.


## 5. Vector Store and Retriever Setup
[back to top ⬆️](#Table-of-contents:)


Now we'll create a vector store to house the document embeddings and enable efficient similarity searches.

- **`Chroma`**: We use ChromaDB as our vector store. It's a lightweight and easy-to-use vector database.
- **`persist_directory`**: This saves the created database to disk, allowing us to reuse it later without re-processing the documents.

In [9]:
# Create a ChromaDB instance to store the document embeddings
from langchain_community.vectorstores import Chroma

vectorstore = Chroma(embedding_function=embedding, persist_directory="./chromadb_varsity", collection_name="varsity_docs")

### Add Documents to the Vector Store
[back to top ⬆️](#Table-of-contents:)


We add the processed document chunks to the vector store. To handle a large number of documents efficiently, we add them in batches. The metadata is also filtered to ensure compatibility with the vector store.

In [ ]:
from langchain_community.vectorstores.utils import filter_complex_metadata


# Function to insert embeddings in batches for a lengthy document set
def add_documents_in_batches(vectorstore, docs, batch_size=100):
    """Adds documents to the vectorstore in batches."""
    for i in range(0, len(docs), batch_size):
        chunk = docs[i : i + batch_size]
        vectorstore.add_documents(chunk)
        print(f"Added batch {i // batch_size + 1}/{(len(docs) - 1) // batch_size + 1}")
    # Persist the database to disk if the method is available
    if hasattr(vectorstore, "persist"):
        vectorstore.persist()


# Filter out complex metadata that might cause issues
filtered_docs = filter_complex_metadata(split_docs)

# Add the documents to the vector store in batches
add_documents_in_batches(vectorstore, filtered_docs)

Added batch 1/1


### Set up a Reranking Retriever
[back to top ⬆️](#Table-of-contents:)


To improve the quality of retrieved documents, we use a reranker. The initial retriever fetches a set of documents (e.g., k=5), and the reranker (`FlashrankRerank`) re-orders them based on their relevance to the query. This ensures that the most relevant context is passed to the LLM.

- **`ContextualCompressionRetriever`**: Wraps a base retriever and a document compressor (the reranker) to create this two-stage retrieval process.

In [11]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import FlashrankRerank

# Set up the base retriever to fetch the top 5 documents
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Initialize the reranker
compressor = FlashrankRerank()

# Create the compression retriever, which combines retrieval and reranking
compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=retriever)

INFO:flashrank.Ranker:Downloading ms-marco-MultiBERT-L-12...
ms-marco-MultiBERT-L-12.zip: 100%|███████████████████████████████████████████████| 98.7M/98.7M [00:02<00:00, 49.8MiB/s]


## 6. Building the RAG Chain
[back to top ⬆️](#Table-of-contents:)


With all the components ready, we now assemble the final RAG pipeline using LangChain's `RetrievalQA` chain. This chain connects the LLM with the retriever.

- **`chain_type="stuff"`**: This means all retrieved documents will be "stuffed" into the prompt sent to the LLM.
- **`return_source_documents=True`**: This is important for evaluation, as it allows us to see which documents were used to generate the answer.

In [12]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=compression_retriever, return_source_documents=True)

## 7. Running the RAG Pipeline
[back to top ⬆️](#Table-of-contents:)


It's time to ask a question! The `qa_chain.invoke` method will execute the full RAG process: retrieve relevant documents, pass them to the LLM along with the question, and return the final answer.

In [13]:
question = "Summarize the document content?"
result = qa_chain.invoke({"query": question})
print("--- Question ---")
print(question)
print("\n--- Answer ---")
print(result["result"])

--- Question ---
Summarize the document content?

--- Answer ---
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

process, along with pre- and post-processing and CODEC stages. This pipeline shows a common process involving style 
transfer of the original video clip, followed by an upscaling operation used to improve the image quality of the newly stylized 
clip. This process is typical of one which might be found in a video editing application.
White Paper | Unlocking the Power of Intel ® Deep Link   Part One: Client Artificial Intelligence Using Intel ® GPUs
Table 1. Device utilization statistics for multi-device test implementation.
* - The theoretical maximum Frames-per-Second (FPS) performance for Deep Link in this configuration would be the sum of the 
performance of each individual GPU (in this case 8.7 + 6.9 = 15.6 FPS), but it is clear from the table that when two

### Extract Answer and Context for Evaluation
[back to top ⬆️](#Table-of-contents:)


For the evaluation step, we need to isolate the generated answer and the source documents (the context or "reference").

In [15]:
answer = result["result"]
context = " ".join([d.page_content for d in result["source_documents"]])

## 8. Evaluation
[back to top ⬆️](#Table-of-contents:)


To assess the quality of our RAG pipeline, we use a custom `OpenVINORAGEvaluator` class. This class uses OpenVINO-optimized models to calculate several key metrics:

- **BLEU & ROUGE**: Measure the overlap between the generated answer and the reference context.
- **BERTScore**: Computes semantic similarity, which is more advanced than simple overlap.
- **Perplexity**: Measures how well a language model (here, phi-4-mini) predicts the generated text. Lower is better.
- **Diversity**: Calculates the variety of tokens in the response.
- **Racial Bias**: Uses a hate speech detection model to check for biased content.

**Note**: The first time you run this, it will download and convert the necessary evaluation models (phi-4-mini and a hate speech model) to the OpenVINO format. This is a one-time setup.

In [ ]:
from notebook_utils import device_widget

eval_device = device_widget()
eval_device

Dropdown(description='Device:', index=3, options=('CPU', 'GPU', 'NPU', 'AUTO'), value='AUTO')

In [ ]:
import openvino as ov
import numpy as np
import torch
from transformers import AutoTokenizer
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score
from nltk.util import ngrams
import os
from optimum.intel import OVModelForCausalLM, OVModelForSequenceClassification


class OpenVINORAGEvaluator:
    """OpenVINO-optimized RAG Evaluator"""

    def __init__(self, device=eval_device.value, models_dir="./openvino_models"):
        self.device = device
        self.models_dir = models_dir
        self.core = ov.Core()

        # Initialize models and tokenizers
        self.phi4_model, self.phi4_tokenizer = self.load_phi4_model()
        self.bias_model, self.bias_tokenizer = self.load_bias_model()

        print(f"OpenVINO RAG Evaluator initialized on {device}")

    def convert_phi4_to_openvino(self):
        phi4_path = os.path.join(self.models_dir, "phi4-mini-openvino")
        os.makedirs(phi4_path, exist_ok=True)

        # Use the HuggingFace repo name for Phi-4-mini
        model_id = "microsoft/Phi-4-mini-instruct"

        ov_model = OVModelForCausalLM.from_pretrained(model_id, export=True, compile=False)
        ov_model.save_pretrained(phi4_path)
        print(f"Phi-4-mini converted and saved to {phi4_path}")

    def convert_bias_model_to_openvino(self):
        """Convert hate speech model to OpenVINO format"""
        print("Converting hate speech model to OpenVINO format...")

        try:
            bias_path = os.path.join(self.models_dir, "hate-speech-openvino")
            os.makedirs(bias_path, exist_ok=True)
            ov_model = OVModelForSequenceClassification.from_pretrained("Hate-speech-CNERG/dehatebert-mono-english", export=True, compile=False)
            ov_model.save_pretrained(bias_path)
            print(f"Hate speech model converted and saved to {bias_path}")
        except ImportError:
            print("optimum[intel] not found. Please install: pip install optimum[intel]")
            raise

    def load_phi4_model(self):
        """Load OpenVINO Phi-4-mini model"""
        phi4_path = os.path.join(self.models_dir, "phi4-mini-openvino")

        # Check if model exists, if not convert it
        if not os.path.exists(os.path.join(phi4_path, "openvino_model.xml")):
            self.convert_phi4_to_openvino()

        try:
            model = OVModelForCausalLM.from_pretrained(phi4_path, device=self.device)
            tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-4-mini-instruct")
            print("Phi-4-mini loaded with OpenVINO backend")
            return model, tokenizer
        except Exception as e:
            print(f"Error loading Phi-4-mini: {e}")

    def load_bias_model(self):
        """Load OpenVINO hate speech detection model"""
        bias_path = os.path.join(self.models_dir, "hate-speech-openvino")

        # Check if model exists, if not convert it
        if not os.path.exists(os.path.join(bias_path, "openvino_model.xml")):
            self.convert_bias_model_to_openvino()

        try:
            model = OVModelForSequenceClassification.from_pretrained(bias_path, device=self.device)
            tokenizer = AutoTokenizer.from_pretrained("Hate-speech-CNERG/dehatebert-mono-english")
            print("Hate speech model loaded with OpenVINO backend")
            return model, tokenizer
        except Exception as e:
            print(f"Error loading hate speech model: {e}")
            # Fallback to manual loading
            return self._load_bias_manual()

    def evaluate_bleu_rouge(self, candidates, references):
        """BLEU and ROUGE evaluation (corrected version)"""
        try:
            # Ensure inputs are lists
            if isinstance(candidates, str):
                candidates = [candidates]
            if isinstance(references, str):
                references = [references]

            # BLEU Score calculation
            # Convert to token lists for BLEU
            candidate_tokens = [candidate.split() for candidate in candidates]
            reference_tokens = [[ref.split()] for ref in references]  # Note: double nested for corpus_bleu

            # Use smoothing to handle edge cases (short sentences)
            smoothing = SmoothingFunction()
            bleu_score = corpus_bleu(reference_tokens, candidate_tokens, smoothing_function=smoothing.method1)

            # ROUGE Score calculation
            scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
            rouge_scores = []

            for ref, cand in zip(references, candidates):
                score = scorer.score(ref, cand)
                rouge_scores.append(score)

            # Average ROUGE-1 F1 score
            rouge1 = sum([score["rouge1"].fmeasure for score in rouge_scores]) / len(rouge_scores)

            return bleu_score, rouge1

        except Exception as e:
            print(f"Error in BLEU/ROUGE calculation: {e}")
            return 0.0, 0.0

    def evaluate_bert_score(self, candidates, references):
        """BERT Score evaluation (unchanged)"""
        P, R, F1 = score(candidates, references, lang="en", model_type="bert-base-multilingual-cased")
        return P.mean().item(), R.mean().item(), F1.mean().item()

    def evaluate_perplexity(self, text):
        """Simplified perplexity calculation"""
        try:
            # Tokenize text
            encodings = self.phi4_tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
            input_ids = encodings.input_ids

            with torch.no_grad():
                # Try to get outputs from model
                if hasattr(self.phi4_model, "forward"):
                    # Optimum Intel model
                    try:
                        outputs = self.phi4_model(input_ids)
                        if hasattr(outputs, "logits"):
                            logits = outputs.logits
                        else:
                            logits = outputs[0]
                    except Exception:
                        return self._fallback_perplexity(text)
                else:
                    # Manual OpenVINO inference
                    return self._fallback_perplexity(text)

                # Calculate perplexity from logits
                # Shift logits and labels for next token prediction
                shift_logits = logits[..., :-1, :].contiguous()
                shift_labels = input_ids[..., 1:].contiguous()

                # Calculate cross entropy loss
                loss_fct = torch.nn.CrossEntropyLoss(reduction="mean")
                shift_logits = shift_logits.view(-1, shift_logits.size(-1))
                shift_labels = shift_labels.view(-1)

                loss = loss_fct(shift_logits, shift_labels)
                perplexity = torch.exp(loss)

                return perplexity.item()
        except Exception as e:
            print(f"Error in perplexity calculation: {e}")

    def evaluate_diversity(self, texts):
        """Diversity evaluation (unchanged)"""
        all_tokens = [tok for text in texts for tok in text.split()]
        unique_bigrams = set(ngrams(all_tokens, 2))
        diversity_score = len(unique_bigrams) / len(all_tokens) if all_tokens else 0
        return diversity_score

    def evaluate_racial_bias(self, text):
        """OpenVINO-optimized bias evaluation"""
        try:
            if hasattr(self.bias_model, "__call__"):
                # Optimum Intel approach
                inputs = self.bias_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

                with torch.no_grad():
                    outputs = self.bias_model(**inputs)
                    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)

                # Assuming label 1 is hate speech (verify based on model)
                bias_score = probabilities[0][1].item()

            else:
                # Manual OpenVINO inference
                bias_score = self._manual_bias_inference(text)

            return bias_score

        except Exception as e:
            print(f"Error in bias evaluation: {e}")
            return 0.0

    def _manual_bias_inference(self, text):
        """Manual bias inference for core OpenVINO model"""
        try:
            # Tokenize input
            inputs = self.bias_tokenizer(text, return_tensors="np", truncation=True, max_length=512, padding=True)

            # Prepare inputs for OpenVINO
            ov_inputs = {}
            for key, value in inputs.items():
                if key in ["input_ids", "attention_mask", "token_type_ids"]:
                    ov_inputs[key] = value.astype(np.int64)

            # Run inference
            outputs = self.bias_model(ov_inputs)

            # Get logits and apply softmax
            logits = outputs[list(outputs.keys())[0]][0]
            exp_logits = np.exp(logits - np.max(logits))
            probabilities = exp_logits / np.sum(exp_logits)

            # Return probability of hate speech (assuming index 1)
            return float(probabilities[1]) if len(probabilities) > 1 else 0.0

        except Exception as e:
            print(f"Manual bias inference error: {e}")
            return 0.0

    def evaluate_all(self, question, response, reference):
        """Comprehensive evaluation using OpenVINO models"""
        candidates = [response]
        references = [reference]

        try:
            bleu, rouge1 = self.evaluate_bleu_rouge(candidates, references)
            bert_p, bert_r, bert_f1 = self.evaluate_bert_score(candidates, references)
            perplexity = self.evaluate_perplexity(response)
            diversity = self.evaluate_diversity(candidates)
            racial_bias = self.evaluate_racial_bias(response)

            return {
                "BLEU": bleu,
                "ROUGE-1": rouge1,
                "BERT P": bert_p,
                "BERT R": bert_r,
                "BERT F1": bert_f1,
                "Perplexity": perplexity,
                "Diversity": diversity,
                "Racial Bias": racial_bias,
            }

        except Exception as e:
            print(f"Error in evaluation: {e}")
            return {"BLEU": 0.0, "ROUGE-1": 0.0, "BERT P": 0.0, "BERT R": 0.0, "BERT F1": 0.0, "Perplexity": float("inf"), "Diversity": 0.0, "Racial Bias": 0.0}


if __name__ == "__main__":
    # First-time setup (converts models)
    print("Setting up OpenVINO RAG Evaluator...")
    OpenVINORAGEvaluator(device=eval_device.value, models_dir="./openvino_models")  # or "GPU" for Intel integrated graphics

Setting up OpenVINO RAG Evaluator...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

INFO:nncf:Statistics of the bitwidth distribution:
+---------------------------+-----------------------------+----------------------------------------+
| Weight compression mode   | % all parameters (layers)   | % ratio-defining parameters (layers)   |
+===========================+=============================+========================================+
| int8_asym, per-channel    | 100% (129 / 129)            | 100% (129 / 129)                       |
+---------------------------+-----------------------------+----------------------------------------+


Output()

Phi-4-mini converted and saved to ./openvino_models\phi4-mini-openvino
Phi-4-mini loaded with OpenVINO backend
Converting hate speech model to OpenVINO format...


...

pytorch_model.bin:   0%|          | 0.00/669M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/152 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Hate speech model converted and saved to ./openvino_models\hate-speech-openvino
Hate speech model loaded with OpenVINO backend
OpenVINO RAG Evaluator initialized on AUTO


### Run the Evaluation
[back to top ⬆️](#Table-of-contents:)


Finally, we initialize the `OpenVINORAGEvaluator` and call `evaluate_all` to get a dictionary of scores. This provides a quantitative look at the performance of our RAG pipeline for the given query.

In [18]:
# Initialize the evaluator (this might take a moment on the first run)
question = "Summarize the document content?"
evaluator = OpenVINORAGEvaluator(device=eval_device.value)

# Prepare the data for evaluation
response_text = answer
reference_text = context

# Get all evaluation metrics
metrics = evaluator.evaluate_all(question, response_text, reference_text)

Phi-4-mini loaded with OpenVINO backend


INFO:absl:Using default tokenizer.


Hate speech model loaded with OpenVINO backend
OpenVINO RAG Evaluator initialized on AUTO


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

In [19]:
print("--- Evaluation Metrics ---")
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

--- Evaluation Metrics ---
BLEU: 0.8286
ROUGE-1: 0.9051
BERT P: 0.9034
BERT R: 0.9253
BERT F1: 0.9142
Perplexity: 5.1875
Diversity: 0.6391
Racial Bias: 0.0592
